# 06 — Advanced RAG and Evaluation

Notebook 05 ended on a failure: the retriever returned a chunk about the planet Mercury when you asked about Project Mercury. The lesson there was that *similarity is not relevance* — and that you can see the gap by looking at the retrieval trace.

This notebook is about closing that gap. Three improvements to retrieval, three failure modes they address, and a small evaluation framework so you can tell whether a change actually helped — or just changed the output.

```
Baseline retrieval        →  fails on keyword collisions, paraphrase, and noise
+ hybrid weighting        →  fixes some keyword cases
+ cross-encoder reranking →  fixes ranking errors in the top-K
+ query expansion         →  fixes paraphrase / vocabulary mismatch
+ evaluation              →  tells you which of the above actually helped on YOUR data
```

By the end of this notebook you should be able to answer two questions about any RAG pipeline: *what's broken?* and *did my fix work?*

**Prerequisites:** notebook 05. Same Ollama models (`qwen3:8b`, `nomic-embed-text-v2-moe`). The cross-encoder reranker step also needs `sentence-transformers` installed — there's a fallback if you don't have it.


## A slightly bigger corpus

The four-document corpus from notebook 05 made the keyword collision visible but it's too small for meaningful evaluation. Let's build an eight-document version that exposes three failure modes deliberately:

- **Two "Mercury" docs** — the keyword collision from notebook 05.
- **Two docs about embeddings** — phrased differently from a likely query.
- **Two docs about deployment topics** — same domain, different wording.
- **Two unrelated noise docs** — chemistry and weather, to give bad retrievals something to grab.

This is the smallest corpus that makes the metrics in this notebook non-trivial.


In [ ]:
import tempfile
from pathlib import Path

corpus_dir = Path(tempfile.mkdtemp(prefix="rag_nb06_"))
print(f"Corpus: {corpus_dir}")

docs = {
    # Keyword collision pair
    "mercury_planet.md": """# Mercury (planet)
Mercury is the smallest and innermost planet in the Solar System. Its
orbital period around the Sun is 88 Earth days. Surface temperatures
swing from 430°C in daylight to -180°C at night because Mercury has
no atmosphere to retain heat.
""",
    "project_mercury_ops.md": """# Project Mercury — Operations Note
Project Mercury is our customer telemetry pipeline. Production runs in
the us-west-2 region. Ingest endpoints sit behind a load balancer at
telemetry.example.internal. Daily volume averages 2.4 billion events.
The on-call rotation is documented in the runbook.
""",
    # Paraphrase / synonym pair
    "embeddings_intro.md": """# What an Embedding Is
An embedding is a vector — a list of numbers — that represents a piece
of text in a way that captures meaning. Texts with similar meaning have
embeddings that are close together in vector space.
""",
    "vector_representation.md": """# Vector Representation of Text
Modern retrieval encodes documents as fixed-length numerical arrays.
Each array — typically 384 to 1024 floats — positions the document in
a learned semantic space. Nearby points mean related content.
""",
    # Domain-overlap pair (both about deployment, different specifics)
    "blue_green_deployment.md": """# Blue/Green Deployment Pattern
Run two identical production environments, blue and green. Only one
serves live traffic at a time. To deploy, push the new version to the
inactive one, validate, then switch traffic. Rollback is one DNS flip.
""",
    "canary_release.md": """# Canary Releases
Roll out a new version to a small percentage of users — typically one
to five percent — and monitor error rates and latency. If the canary
behaves, gradually shift more traffic. This catches regressions
before they affect everyone.
""",
    # Noise pair
    "iron_smelting.md": """# Iron Smelting
Smelting separates iron from its ore using high heat and a reducing
agent, usually carbon monoxide produced by burning coke. The blast
furnace process dates to the medieval period.
""",
    "monsoon_winds.md": """# Monsoon Winds
The Asian monsoon is driven by differential heating between the
continent and the surrounding oceans. Summer monsoons bring moist
oceanic air over the warm land; winter monsoons reverse the flow.
""",
}
for name, text in docs.items():
    (corpus_dir / name).write_text(text)
print(f"Wrote {len(docs)} files.")


## Baseline pipeline

Start with the same defaults from notebook 05. No reranker, no expander. Hybrid retrieval is on by default in `rag_lib`, but with a small `bm25_weight` (0.4) it leans toward the dense (semantic) side.


In [ ]:
from rag_lib import RAGPipeline

scratch = Path(tempfile.mkdtemp(prefix="rag_nb06_store_"))

base_config = {
    "embedder": {
        "host": "http://localhost:11434",
        "model": "nomic-embed-text-v2-moe",
        "timeout": 120,
    },
    "storage": {
        "backend": "chromadb",
        "path": str(scratch / "chroma"),
        "collection_prefix": "rag_nb06_baseline_",
        "bm25_path": str(scratch / "bm25_baseline"),
    },
    "chunker": {
        "max_embed_tokens": 1800,
        "defaults": {"strategy": "fixed_size", "chunk_size": 512, "chunk_overlap": 50},
        "doc_types": {"guide": {"strategy": "sentence_window", "window_size": 3}},
    },
    "retriever": {
        "n_results": 5,
        "max_context_tokens": 2000,
        "bm25_weight": 0.4,
    },
}

baseline = RAGPipeline(config=base_config)

for path in sorted(corpus_dir.glob("*.md")):
    baseline.ingest(path, doc_type="guide")

print(f"Baseline collections: {baseline.list_collections()}")
print(f"Default collection has {baseline.collection_info('default')['n_chunks']} chunks.")


Quick check — run a query you know the answer to.


In [ ]:
trace = baseline.inspect_query("What region does Project Mercury run in?")

print("Top 5 fused results:")
for doc in trace.fused_results[:5]:
    print(f"  score={doc.score:.3f}  source={doc.source}")


On this small corpus the baseline often gets the correct doc in position 1 or 2. On real corpora with hundreds of "Mercury" mentions, it doesn't. The improvements that follow are what keep retrieval honest at scale — and the evaluation block at the end of this notebook is how you tell the difference.


## Improvement 1 — Tuning the hybrid weight

`rag_lib` already does hybrid retrieval: it runs both dense (embedding) search and BM25 (keyword) search, then fuses the two ranked lists with reciprocal rank fusion (RRF). The `bm25_weight` config knob controls the balance.

| `bm25_weight` | Behavior |
|---|---|
| 0.0 | Dense only — semantic, paraphrase-friendly, misses exact terms |
| 0.4 | Default — leans semantic, keeps keyword as a backstop |
| 0.7 | Leans keyword — better for proper nouns, code identifiers, exact phrases |
| 1.0 | BM25 only — keyword search, paraphrase-blind |

What RRF actually does, in plain English: for each candidate document, it adds up `1 / (k + rank)` across the two retrievers (k is a small constant). A document that ranks well in both lists ends up on top. A document that ranks well in just one only ends up on top if it ranks *very* well there.

Let's set up a keyword-leaning pipeline alongside the baseline and compare.


In [ ]:
keyword_config = {**base_config}
keyword_config["storage"] = {
    "backend": "chromadb",
    "path": str(scratch / "chroma"),
    "collection_prefix": "rag_nb06_keyword_",
    "bm25_path": str(scratch / "bm25_keyword"),
}
keyword_config["retriever"] = {**base_config["retriever"], "bm25_weight": 0.7}

keyword_leaning = RAGPipeline(config=keyword_config)
for path in sorted(corpus_dir.glob("*.md")):
    keyword_leaning.ingest(path, doc_type="guide")

# Side-by-side on a proper-noun query
query = "What region does Project Mercury run in?"

print(f"Query: {query}\n")
print("baseline (bm25_weight=0.4):")
for doc in baseline.inspect_query(query).fused_results[:3]:
    print(f"  score={doc.score:.3f}  source={doc.source}")
print("\nkeyword-leaning (bm25_weight=0.7):")
for doc in keyword_leaning.inspect_query(query).fused_results[:3]:
    print(f"  score={doc.score:.3f}  source={doc.source}")


On a small corpus, both configurations usually rank the operations note first. The interesting case is when they diverge — typically when a query uses a proper noun that BM25 weights highly but dense embeddings treat as just another token. The keyword-leaning config tends to win on identifiers and codenames; the dense-leaning baseline tends to win on conceptual queries.

The right value of `bm25_weight` is a property of your corpus, not a universal best. The evaluation block at the end of this notebook is how you find it.


## Improvement 2 — Cross-encoder reranking

Hybrid retrieval gives you a ranked list of candidates. The ranking is approximate — it scores documents *independently* (against the query embedding or against the BM25 statistics), not *jointly*.

A cross-encoder reranker fixes this. It takes the top-K candidates from stage 1 and scores each `(query, chunk)` pair as a unit, looking at them together. Far more accurate than independent scoring; far more expensive too, which is why it's stage 2: applied only to the top 20–50 candidates, not the full corpus.

`rag_lib`'s `CrossEncoderReranker` uses a small MS-MARCO model by default (~80MB, runs on CPU in well under a second).


In [ ]:
try:
    rerank_config = {**base_config}
    rerank_config["storage"] = {
        "backend": "chromadb",
        "path": str(scratch / "chroma"),
        "collection_prefix": "rag_nb06_rerank_",
        "bm25_path": str(scratch / "bm25_rerank"),
    }
    rerank_config["reranker"] = {
        "enabled": True,
        "model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "device": "auto",
    }

    reranked = RAGPipeline(config=rerank_config)
    for path in sorted(corpus_dir.glob("*.md")):
        reranked.ingest(path, doc_type="guide")

    reranker_available = True
    print("Reranker loaded.")
except ImportError as e:
    print(f"sentence-transformers not installed — skipping reranker demo: {e}")
    print("Install with: pip install sentence-transformers")
    reranker_available = False


In [ ]:
if reranker_available:
    query = "How do you ship a new release without breaking users?"

    print(f"Query: {query}\n")

    print("baseline (no reranker):")
    for doc in baseline.inspect_query(query).fused_results[:5]:
        print(f"  score={doc.score:.3f}  source={doc.source}")

    print("\nwith reranker:")
    trace = reranked.inspect_query(query)
    for doc in trace.selected_results[:5]:
        print(f"  score={doc.score:.3f}  source={doc.source}")

    print(f"\nTrace events (reranker stage):")
    for event in trace.events:
        if "rerank" in event.get("event_type", "").lower():
            print(f"  {event}")
else:
    print("Skipped — reranker not available.")


Two things to notice:

- **The scores change scale.** Stage 1 scores are RRF-fused rankings (cosine-similarity-ish). Reranker scores are cross-encoder logits — a different number entirely. Don't compare them numerically across stages.
- **The ordering can change.** This is the whole point. The reranker takes joint context into account, so a candidate that ranked third in stage 1 may rank first after stage 2 — usually because it actually answers the question while higher-ranked candidates just contain the keywords.

When is reranking worth the cost? When you care about the *order* of the top few results, not just whether the right document is somewhere in the top 20. For most production RAG, that's always.


## Improvement 3 — Query expansion

Sometimes the problem is the query itself. A user asks "How do you ship a new release without breaking users?" but the relevant chunk talks about "canary releases" and "blue/green deployment." The keywords don't match. The embedding match may be weak. Retrieval misses entirely.

Query expansion attacks this by generating *additional* phrasings of the query and running retrieval on each variant. `rag_lib`'s `QueryExpander` asks an LLM (small one — `qwen3:8b`) for three alternative phrasings, then `inspect_query()` retrieves against all four (original + three variants) and fuses the results.

This is the "HyDE-flavored" expansion the literature talks about. Pure HyDE generates a hypothetical *answer* and embeds that; this implementation generates alternative *questions*. Both work; alternative-questions is faster and more robust on small models.


In [ ]:
expander_config = {**base_config}
expander_config["storage"] = {
    "backend": "chromadb",
    "path": str(scratch / "chroma"),
    "collection_prefix": "rag_nb06_expand_",
    "bm25_path": str(scratch / "bm25_expand"),
}
expander_config["expander"] = {
    "enabled": True,
    "model": "qwen3:8b",
    "cache_size": 256,
}

expanded = RAGPipeline(config=expander_config)
for path in sorted(corpus_dir.glob("*.md")):
    expanded.ingest(path, doc_type="guide")

# Use a question whose wording differs from the documents
query = "How do you ship a new release without breaking users?"
trace = expanded.inspect_query(query)

print(f"Query: {query}")
print(f"Variants generated: {trace.query_variants}")
print(f"\nTop 5 fused across all variants:")
for doc in trace.fused_results[:5]:
    print(f"  score={doc.score:.3f}  source={doc.source}")


When expansion works, you'll see the canary and blue/green docs rank higher than they did in the baseline — because some variant of the query contained the right vocabulary.

When expansion *fails*, it does so loudly: the small model generates variants that are unrelated to the original question, and retrieval pulls in noise. The expander has a fallback that just returns the original query on errors, but a model generating off-topic variants isn't an error — it's a silent quality regression. This is why expansion is off by default in `rag_lib`, and why you need evaluation before turning it on in production.


## The evaluation problem

Three improvements, three different failure modes addressed. But how do you know any of them actually helped on *your* data?

The wrong way: eyeball a few queries and trust your gut. This is how teams ship regressions — every change feels like an improvement to the person who made it.

The right way: a small held-out set of queries with known answers, run before *and* after each change, scored on metrics that diagnose *what's wrong* — not just *how good* the output is.

The three diagnostic metrics most RAG evaluation frameworks use:

| Metric | Tells you | Where it points |
|---|---|---|
| **Context Recall** | of all relevant chunks, what fraction did you retrieve? | Low → retrieval is missing things (try query expansion, broader candidate set) |
| **Context Precision** | of retrieved chunks, what fraction is relevant? | Low → retrieval is bringing in noise (try reranking, tighter top-k) |
| **Faithfulness** | does the generated answer actually use the retrieved context? | Low → the model is hallucinating (try a better grounding prompt, low temperature, or a different model) |

Each metric points to a different fix. Reporting only an overall score collapses the diagnosis.


## A tiny ground-truth dataset

For a corpus this small, ground truth is straightforward: for each test question, name the document(s) that contain the answer.

Real evaluation datasets have hundreds of questions, may have multiple relevant docs per query, and often grade chunks (not docs) by relevance. The principle is the same.


In [ ]:
ground_truth = [
    {
        "question": "What region does Project Mercury run in?",
        "relevant_sources": ["project_mercury_ops.md"],
        "failure_mode": "keyword_collision",
    },
    {
        "question": "How does an embedding represent text?",
        "relevant_sources": ["embeddings_intro.md", "vector_representation.md"],
        "failure_mode": "paraphrase",
    },
    {
        "question": "How do you ship a new release without breaking users?",
        "relevant_sources": ["blue_green_deployment.md", "canary_release.md"],
        "failure_mode": "vocabulary_mismatch",
    },
    {
        "question": "What's the orbital period of the planet Mercury?",
        "relevant_sources": ["mercury_planet.md"],
        "failure_mode": "keyword_collision",
    },
    {
        "question": "What causes monsoon winds?",
        "relevant_sources": ["monsoon_winds.md"],
        "failure_mode": "easy_baseline",
    },
]
print(f"{len(ground_truth)} test queries.")


## Manual evaluation

You don't need a framework to compute Context Recall and Context Precision on a small dataset. The formulas are arithmetic:

```
context_recall    = |retrieved ∩ relevant| / |relevant|
context_precision = |retrieved ∩ relevant| / |retrieved|
```

Faithfulness needs an LLM judge — we'll come back to that.


In [ ]:
def evaluate_pipeline(pipeline, ground_truth, top_k=5, label=""):
    """Compute mean Context Recall and Precision across a ground-truth set."""
    recalls, precisions = [], []
    by_failure = {}

    for entry in ground_truth:
        question = entry["question"]
        relevant = set(entry["relevant_sources"])

        # Retrieve and extract source filenames
        chunks = pipeline.retrieve(question)
        retrieved = {Path(c.source_id).name for c in chunks[:top_k]}

        hits = retrieved & relevant
        recall    = len(hits) / len(relevant)       if relevant else 0.0
        precision = len(hits) / len(retrieved)      if retrieved else 0.0

        recalls.append(recall)
        precisions.append(precision)
        by_failure.setdefault(entry["failure_mode"], []).append(recall)

    print(f"=== {label or 'pipeline'} ===")
    print(f"  Context Recall    @ top-{top_k}: {sum(recalls)/len(recalls):.3f}")
    print(f"  Context Precision @ top-{top_k}: {sum(precisions)/len(precisions):.3f}")
    print(f"  By failure mode (recall):")
    for mode, vals in by_failure.items():
        print(f"    {mode:22s}: {sum(vals)/len(vals):.3f}")
    return {"recall": sum(recalls)/len(recalls), "precision": sum(precisions)/len(precisions)}

baseline_scores = evaluate_pipeline(baseline, ground_truth, label="baseline")
print()
keyword_scores = evaluate_pipeline(keyword_leaning, ground_truth, label="keyword-leaning (bm25=0.7)")
print()
if reranker_available:
    rerank_scores = evaluate_pipeline(reranked, ground_truth, label="with reranker")
    print()
expand_scores = evaluate_pipeline(expanded, ground_truth, label="with query expansion")


The output gives you four numbers per pipeline — overall recall, overall precision, and a recall breakdown by failure mode. That breakdown is the whole point of the per-question annotation: when a change moves the overall number, the per-mode breakdown tells you *which queries it helped and which it didn't*.

Some patterns you may see:

- The reranker often improves precision without changing recall much, because it doesn't fetch different candidates — it just reorders them.
- Query expansion often improves recall on the `vocabulary_mismatch` queries while leaving everything else roughly the same. If it doesn't, your expander LLM is generating bad variants — try `qwen3:14b` or a different prompt.
- Keyword-leaning hybrid weights help on `keyword_collision` and may hurt on `paraphrase`. There's no universal best.


## Faithfulness — needs an LLM judge

Recall and precision measure retrieval. They say nothing about whether the model actually grounds its answer in what it retrieved. A pipeline can have perfect retrieval and still hallucinate.

Faithfulness is harder to measure mechanically. The standard approach is to ask another LLM:

> Here is the question, the retrieved context, and the generated answer.
> Is every claim in the answer supported by the context? Score 0.0 to 1.0.

This is what RAGAS does. It's also why you should never use the same model as both generator and judge — you measure self-consistency, not quality.

`rag_lib` has the eval scaffolding wired up — `load_ground_truth()`, `EvalReport`, `print_report()` — but the actual `run_eval()` call that integrates RAGAS is currently a stub:

```python
from rag_lib.eval.ragas_runner import run_eval, load_ground_truth, print_report

# This API exists and the arguments are real, but the implementation
# currently raises NotImplementedError — wiring it up is on the roadmap.
```

For now, the manual evaluation above gets you 80% of the way. When `run_eval()` lands, the same ground-truth file format will work — you'll just gain the faithfulness metric and a judge-LLM-based scoring loop.


## The optimize-evaluate loop

There's an order-of-operations rule worth internalizing:

1. **Build evaluation first**, before any optimization. Even a small ground-truth set is enough.
2. **Run the baseline** through it. Record the numbers.
3. **Change one thing.** One config knob, one component, one prompt edit.
4. **Re-run evaluation.** If the metric you cared about went up *and* nothing else regressed, keep the change.
5. **Go to 3.**

This is what people call the "Karpathy autoresearch loop" or the "Ralph loop" — eval-driven iteration. The principle is mundane: changes that aren't measured aren't improvements. The discipline is the hard part. It is tempting to ship a change because three test queries look better and the demo is on Friday. Don't. The three test queries are not your evaluation set.

One more property of this loop: changes that **don't** show up in the metric are also informative. If reranking changes the output on every query but doesn't move precision, your evaluation set is too coarse — the reranker is rearranging chunks within the relevant set, which doesn't change a set-membership metric. Make your eval set richer (per-rank scoring, or relevance grades 0–3) when this happens.


## Learning checkpoint

1. Three retrieval improvements were introduced. For each, name a failure mode it addresses and one where it doesn't help (or hurts).
2. Why is reranking a *second stage* rather than a *replacement* for the first stage?
3. Context Recall, Context Precision, Faithfulness — which one needs an LLM judge to compute? Why?
4. Why is it a problem to use the same model as both generator and judge in RAG evaluation?
5. You change the `bm25_weight` from 0.4 to 0.7 and the overall Context Recall goes up. What additional check do you want to do before keeping the change?

## What's next

- **Notebook 07** is the reference-app walkthrough — `language_tutor` shows how these components compose in a real application.
- **Notebook 09** generalizes this optimize-evaluate loop using the shared evaluator contracts.

## Clean up

```python
import shutil
shutil.rmtree(corpus_dir, ignore_errors=True)
shutil.rmtree(scratch, ignore_errors=True)
```
